# Frontload-cl Colab smoke (1×A100)

Checks whether **one A100** can hold the real per-rank microbatch (`24×4096`) for OLMo2-370M with FlashAttention-2 and `torch.compile`.

This is **not** the platform 8×A100 run and does **not** exercise the primer/control curriculum or `s3://edullm-data`.

**Runtime → Change runtime type → GPU** (A100 if you have Pro). Then run all cells.

Repo path used below: `/content/OLMo-core` on the branch `edullm/frontload-cl`.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type)"
props = torch.cuda.get_device_properties(0)
print(f"device: {props.name}")
print(f"memory: {props.total_memory / 1024**3:.1f} GiB")
print(f"capability: {props.major}.{props.minor}")
print(f"torch: {torch.__version__}  cuda: {torch.version.cuda}")
if props.major < 8:
    print("WARNING: FlashAttention-2 wants SM80+ (A100). Use --attn-backend torch on older GPUs.")

## 2. Clone the branch

Uses a public HTTPS clone. **Re-run this cell whenever the branch moves** — if `/content/OLMo-core` already exists it fetches and `git reset --hard origin/edullm/frontload-cl`.

If your notebook still prints `already present:` you are on an **old notebook copy**. Paste this in a fresh cell instead:

```python
%cd /content/OLMo-core
!git fetch --depth 1 origin edullm/frontload-cl
!git checkout -B edullm/frontload-cl origin/edullm/frontload-cl
!git reset --hard origin/edullm/frontload-cl
!git log -1 --oneline
```

Or Runtime → Disconnect and delete runtime, upload/open the notebook from the branch tip, then run all.


In [ ]:
from pathlib import Path

REPO = Path("/content/OLMo-core")
BRANCH = "edullm/frontload-cl"
REMOTE = "https://github.com/edu-llm/OLMo-core.git"

if (REPO / ".git").is_dir():
    %cd {REPO}
    !git remote set-url origin {REMOTE}
    !git fetch --depth 1 origin {BRANCH}
    !git checkout -B {BRANCH} origin/{BRANCH}
    !git reset --hard origin/{BRANCH}
    !git clean -fd
else:
    !git clone --depth 1 --branch {BRANCH} {REMOTE} {REPO}
    %cd {REPO}

print("HEAD after sync:")
!git rev-parse HEAD
!git log -1 --oneline
!grep -n "FRONTLOAD_CL_MICROBENCH_V4\|fused_linear_cross_entropy_loss" .edullm/frontload_cl/colab_smoke.py src/olmo_core/nn/functional/cross_entropy_loss.py | head


## 3. Install OLMo-core + FlashAttention-2 + Liger (match the platform)

Colab often ships a newer torch (you may see `2.11+cu128`) that has **no** official FA2 wheel. Without FA2, SDPA at the real microbatch (`24×4096`) typically **OOMs on a 40 GiB A100**.

Also: default CE builds a full **fp32 logits** tensor `(B·T·V) ≈ 39 GiB` at this shape. Train scripts and the microbench use **fused linear CE** (`liger-kernel`) so that never allocates — without it the platform 8×40 GiB run would OOM too.

This cell:
1. Pins **torch 2.9** (same major as the eduLLM image)
2. Installs OLMo-core editable (`--no-deps`) + **liger-kernel**
3. Installs the **Dao FA 2.8.3 wheel** for torch2.9 / cu12 / cp312

After pinning, **Restart session**, then continue from the GPU check.

In [ ]:
import importlib
import subprocess
import sys

import torch

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

print("Colab torch before pin:", torch.__version__, "cuda:", torch.version.cuda)

# Platform image pins torch 2.9; FA2 prebuilt wheels match that, not Colab's default 2.11.
if not torch.__version__.startswith("2.9."):
    print("Pinning torch==2.9.0+cu128 (and matching torchvision/torchaudio)…")
    pip(
        "torch==2.9.0+cu128",
        "torchvision==0.24.0+cu128",
        "torchaudio==2.9.0+cu128",
        "--index-url",
        "https://download.pytorch.org/whl/cu128",
    )
    print("Restart the session now (Runtime → Restart session), then re-run from cell 1.")
    raise SystemExit("torch pin installed; restart session, then continue from the GPU check")

# Editable install without the kitchen-sink `all` extra.
pip("-e", ".", "--no-deps")
pip("numpy", "rich", "cached-path", "safetensors", "dataclass-extensions", "bettermap", "pandas")
# Required for LMLossImplementation.fused_linear (avoids ~39GiB fp32 logits at 24×4096).
pip("liger-kernel")

ATTN = "flash_2"
try:
    import flash_attn  # noqa: F401
    print("flash_attn already importable:", flash_attn.__version__)
except Exception:
    # Same URL pattern as .edullm/Dockerfile — never fall back to a source compile.
    abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheel = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
        f"flash_attn-2.8.3+cu12torch2.9cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
    )
    print("installing", wheel)
    try:
        pip(wheel, "--no-build-isolation", "--no-cache-dir")
        import flash_attn
        print("installed flash_attn", flash_attn.__version__)
    except Exception as exc:
        print("FA2 wheel install failed:", type(exc).__name__, exc)
        print(
            "Do NOT microbench with --attn-backend torch at 24×4096 on 40GiB — "
            "SDPA will OOM and that does not predict platform flash_2 memory."
        )
        ATTN = "unavailable"

print("ATTN_BACKEND =", ATTN)
assert ATTN == "flash_2", "Need flash_2 for a meaningful A100 microbench; fix the wheel install first"
importlib.import_module("liger_kernel.ops.fused_linear_cross_entropy")
print("liger-kernel fused CE import ok")


## 4. Microbench (the important cell)

Same shape as one rank on `gpu-8xa100`: **24 sequences × 4096**, FA2, bf16.

**Restart the session first** if you already OOMed — fragmentation often leaves almost no free memory.

Defaults are stricter about fairness than before: **no** `torch.compile`, **no** full-model Adam (platform HSDP shards those). This is an activation-memory proxy.

```bash
!git -C /content/OLMo-core pull
!python .edullm/frontload_cl/colab_smoke.py microbench --steps 3 --attn-backend flash_2
```

Success = a few steps complete; note `peak_mem_gib`.

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# You MUST see FRONTLOAD_CL_MICROBENCH_V4_FUSED_CE_AC and ac=True in the output.
# V3 still OOMs: fused CE alone is not enough without activation checkpointing.
!git rev-parse --short HEAD
!git log -1 --oneline
!python .edullm/frontload_cl/colab_smoke.py gpu-info
!python .edullm/frontload_cl/colab_smoke.py microbench --steps 3 --attn-backend {ATTN}


## 5. Optional: synthetic data → short Trainer fit

Exercises composable data loader + `TransformerTrainModule` on a **flat** mix (not primer/control). Skip if the microbench already answered your question.

In [ ]:
!python .edullm/frontload_cl/colab_smoke.py write-data --out /content/frontload-synth
!python .edullm/frontload_cl/colab_smoke.py train --data /content/frontload-synth --steps 5 --attn-backend {ATTN}

## How to read the result

| Outcome | Meaning |
| --- | --- |
| `ok: true` with `attn_backend: flash_2`, peak mem well under ~40 GiB | Single-rank shape looks fine for A100 (same microbatch as one platform rank) |
| FA2 install failed / refused | Fix torch pin + FA wheel; **do not** treat SDPA OOM as a platform failure |
| CUDA OOM **with flash_2** | Real concern for the 8-GPU microbatch — investigate before full submit |
| CUDA OOM with `torch` SDPA | Expected on 40 GiB at 24×4096; not informative for the platform image |

Still needed later on the platform: 8-way HSDP/NCCL, real `frontload-cl-10b-v1`, and `.edullm/run-smoke.yaml`.